<a href="https://colab.research.google.com/github/SAMSERVICA/Ai-with-Chucky-Colab-Notebooks/blob/main/Qwen_Image_2_1_NeoTenet_Edition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen-Image-2.1 Precision Image Editing

💎 Brought to you by **NeoTenet**

| Component | Specification |
|---|---|
| **Diffusion Engine** | Qwen-Image-2.1 INT8 |
| **Text/Vision Encoder** | Qwen3-VL 8B INT8 |
| **Conditioning** | Up to 10 Reference Images |
| **Acceleration** | Triton + xFormers |

---

### Architecture & Conditioning Mechanics

Qwen-Image-2.1 is an open-weight foundation model designed for instruction-based image editing and generation. It operates on a 32-layer Single-Stream Diffusion Transformer (DiT) architecture where text prompt tokens and visual latent tokens are concatenated and processed through shared self-attention layers.

* **4-Channel Alpha VAE:** Features a four-channel variational autoencoder capable of directly generating and modifying RGBA images with native transparency, bypassing separate foreground matting steps.
* **Slot 1 (`<image1>`):** The primary target canvas. The model anchors its spatial layout, perspective, and untargeted regions to this image.
* **Slots 2 to 10 (`<image2>` to `<image10>`):** Auxiliary reference donors. The model extracts specified attributes (garments, backgrounds, accessories, lighting, or facial identity) and synthesizes them into `<image1>`.
* **Single-Image Processing ($N = 1$):** When only Slot 1 is populated, instructions operate directly without index tags (e.g., background removal, style transfer, or lighting shifts).
* **Computational Acceleration:** The pipeline utilizes INT8 ConvRot matrix quantization, native hardware FP16 VAE decoding, and automatic reference token downscaling to 768px to prevent quadratic attention expansion during sampling.

In [ ]:
# @title 1. Environment Configuration & Model Initialization (NeoTenet Edition)
# @markdown Configures CUDA memory allocators, downloads INT8 models via aria2, and starts backend execution daemon.

import os
import sys
import time
import urllib.request
import subprocess
import torch

print("═" * 65)
print("💎 NEOTENET STUDIO - ENVIRONMENT & HARDWARE CONFIGURATION")
print("═" * 65)

if not torch.cuda.is_available():
    raise SystemError("⚠️ GPU acceleration required! Go to Runtime > Change runtime type > GPU (T4).")

# پیکربندی حافظه و تخصیص‌دهنده CUDA جهت جلوگیری از تکه‌تکه شدن رم گرافیک
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"• Active GPU      : {gpu_name}")
print(f"• Allocated VRAM  : {vram_gb:.2f} GB")
print(f"• CUDA Allocator  : expandable_segments enabled (Dual-RAM ready)")

%cd /content
if not os.path.exists("/content/ComfyUI"):
    print("\n• Initializing core runtime engine...")
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q -U aria2 pillow websocket-client flask flask-cors pinggy comfy-kitchen triton xformers

# نصب نودهای کمکی برای ورک‌فلوهای پیشرفته
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("rgthree-comfy"):
    !git clone -q https://github.com/rgthree/rgthree-comfy.git
%cd /content/ComfyUI

os.makedirs("/content/ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("/content/ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)
os.makedirs("/content/ComfyUI/input", exist_ok=True)

print("\n📥 Downloading Qwen-Image-2.1 INT8 checkpoints via multi-thread...")

# 1. مدل انتشار Diffusion پایه بدون سانسور (INT8 ConvRot ~7.26 GB)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors" \
  -d /content/ComfyUI/models/diffusion_models -o qwen_image_2.1_int8_convrot.safetensors

# 2. انکودر متنی و بینایی Qwen3-VL 8B INT8 ConvRot (~9.35 GB)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors" \
  -d /content/ComfyUI/models/text_encoders -o qwen3vl_8b_int8_convrot.safetensors

# 3. دیکودر 4-کاناله VAE با پشتیبانی کامل از Alpha/Transparency (~676 MB)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors" \
  -d /content/ComfyUI/models/vae -o qwen_image_2.1_vae_bf16.safetensors

# 4. تصاویر دمو جهت مقداردهی اولیه اسلات‌ها
!wget -q -nc -P /content/ComfyUI/input \
  "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/input/portrait_model_denim.png"
!wget -q -nc -P /content/ComfyUI/input \
  "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/input/clothing_light_blue_denim_shirt.png"
!wget -q -nc -P /content/ComfyUI/input \
  "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/input/angry_broccoli.png"
!wget -q -nc -O /content/ComfyUI/input/scene_tokyo_street.png \
  "https://raw.githubusercontent.com/QwenLM/Qwen-Image/main/assets/demo_images/demo_bg.png" || true

# بستن پورت‌های قبلی
!fuser -k 8188/tcp > /dev/null 2>&1

print("\n⚙️ Starting Dual-Memory ComfyUI backend engine...")
engine_log = open("/content/engine.log", "w")
proc = subprocess.Popen([
    sys.executable, "/content/ComfyUI/main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--fp16-vae",
    "--preview-method", "none"
], stdout=engine_log, stderr=subprocess.STDOUT)

print("Connecting to backend engine...", end="")
for _ in range(90):
    if proc.poll() is not None:
        print("\n❌ Backend engine process terminated unexpectedly!")
        with open("/content/engine.log", "r") as f:
            print(f.read())
        raise RuntimeError("Backend engine crashed during boot. See log above.")

    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as resp:
            if resp.status == 200:
                print(" ✅ Ready!")
                break
    except Exception:
        time.sleep(1)
        print(".", end="", flush=True)
else:
    print("\n❌ Boot timed out. Printing /content/engine.log:")
    with open("/content/engine.log", "r") as f:
        print(f.read())
    raise TimeoutError("Backend engine failed to boot within 90 seconds.")

print("\n" + "═" * 65)
print("🎉 SETUP COMPLETE! Run Cell 2 to launch your NeoTenet Studio.")
print("═" * 65)

═════════════════════════════════════════════════════════════════
💎 NEOTENET STUDIO - ENVIRONMENT & HARDWARE CONFIGURATION
═════════════════════════════════════════════════════════════════
• Active GPU      : Tesla T4
• Allocated VRAM  : 14.56 GB
• CUDA Allocator  : expandable_segments enabled (Dual-RAM ready)
/content
/content/ComfyUI
/content/ComfyUI/custom_nodes
/content/ComfyUI

📥 Downloading Qwen-Image-2.1 INT8 checkpoints via multi-thread...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
aa1606|OK  |       0B/s|/content/ComfyUI/models/diffusion_models/qwen_image_2.1_int8_convrot.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
c5828b|OK  |       0B/s|/content/ComfyUI/models/text_encoders/qwen3vl_8b_int8_convrot.safetensors

Status Legend:
(OK):download completed.

In [ ]:
# @title 2. Launch Interface & Public Tunnel (NeoTenet Edition)
# @markdown Initializes editing interface with dynamic multi-reference slot management. Branded for NeoTenet.

import os
import sys
import json
import time
import glob
import uuid
import base64
import logging
import threading
import subprocess
import urllib.request
import websocket
from PIL import Image
from flask import Flask, request, jsonify, Response, send_from_directory
from flask_cors import CORS
from IPython.display import display, HTML

# آزادسازی پورت 5000 از نشست‌های قبلی
!fuser -k 5000/tcp > /dev/null 2>&1
time.sleep(1)

try:
    import pinggy
except ImportError:
    !pip install -q pinggy
    import pinggy

log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

app = Flask(__name__)
CORS(app)

def optimize_donor_image(filename, max_dim=768):
    """Downscales reference donors to 768px to prevent DiT sequence token explosion."""
    filepath = os.path.join('/content/ComfyUI/input', filename)
    if not os.path.exists(filepath):
        return filename
    try:
        with Image.open(filepath) as img:
            if max(img.size) > max_dim:
                img.thumbnail((max_dim, max_dim), Image.Resampling.LANCZOS)
                opt_name = f"opt_{max_dim}_{filename}"
                opt_path = os.path.join('/content/ComfyUI/input', opt_name)
                img.save(opt_path)
                return opt_name
    except Exception as e:
        print(f"[Warning] Failed to optimize donor: {e}", flush=True)
    return filename

HTML_TEMPLATE = r'''<!DOCTYPE html>
<html lang="en" data-theme="light">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>NeoTenet - Frosted Glass Studio</title>
  <script src="https://cdn.tailwindcss.com"></script>
  <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
  <style>
    /* استایل شیشه‌ای مات داشبورد تصویر ۱ */
    html[data-theme="light"] {
      --bg-wallpaper: linear-gradient(135deg, #bfe3fc 0%, #d8edfc 35%, #fce7db 70%, #fbd5c3 100%);
      --glass-panel: rgba(255, 255, 255, 0.65);
      --glass-border: rgba(255, 255, 255, 0.85);
      --glass-shadow: 0 20px 50px -10px rgba(14, 116, 144, 0.08), inset 0 1px 2px 0 rgba(255, 255, 255, 0.95);
      --card-white: rgba(255, 255, 255, 0.88);
      --card-border: rgba(255, 255, 255, 0.95);
      --text-main: #1e293b;
      --text-sub: #64748b;
      --accent-grad: linear-gradient(135deg, #0284c7 0%, #38bdf8 100%);
    }

    html[data-theme="dark"] {
      --bg-wallpaper: radial-gradient(circle at 80% 20%, #2a1b3d 0%, #110e1e 60%, #08060f 100%);
      --glass-panel: rgba(24, 20, 42, 0.72);
      --glass-border: rgba(255, 255, 255, 0.12);
      --glass-shadow: 0 20px 50px -10px rgba(0, 0, 0, 0.45), inset 0 1px 1px 0 rgba(255, 255, 255, 0.15);
      --card-white: rgba(16, 13, 28, 0.7);
      --card-border: rgba(255, 255, 255, 0.08);
      --text-main: #f8fafc;
      --text-sub: #94a3b8;
      --accent-grad: linear-gradient(135deg, #0284c7 0%, #a855f7 100%);
    }

    body, .frosted-panel, .frosted-card, input, select, textarea {
      transition: background-color 0.22s cubic-bezier(0.16, 1, 0.3, 1),
                  border-color 0.22s cubic-bezier(0.16, 1, 0.3, 1),
                  color 0.2s ease,
                  box-shadow 0.22s cubic-bezier(0.16, 1, 0.3, 1);
    }

    body {
      background: var(--bg-wallpaper);
      background-attachment: fixed;
      min-height: 100vh;
      color: var(--text-main);
      font-family: ui-sans-serif, system-ui, -apple-system, sans-serif;
    }

    .frosted-panel {
      background: var(--glass-panel);
      backdrop-filter: blur(35px) saturate(180%);
      -webkit-backdrop-filter: blur(35px) saturate(180%);
      border: 1px solid var(--glass-border);
      box-shadow: var(--glass-shadow);
      border-radius: 28px;
    }

    .frosted-card {
      background: var(--card-white);
      border: 1px solid var(--card-border);
      box-shadow: 0 4px 20px -2px rgba(0, 0, 0, 0.03);
      border-radius: 20px;
    }

    /* دکمه‌های کپسولی ۳ بعدی رنگی */
    .btn-pill-3d {
      border-radius: 14px;
      font-weight: 800;
      padding: 7px 15px;
      font-size: 12px;
      display: inline-flex;
      align-items: center;
      gap: 6px;
      user-select: none;
      cursor: pointer;
      transition: all 0.16s cubic-bezier(0.34, 1.56, 0.64, 1);
      border-top: 1px solid rgba(255, 255, 255, 0.45);
    }
    .btn-pill-3d:hover {
      transform: translateY(-2px);
      filter: brightness(1.1);
    }
    .btn-pill-3d:active {
      transform: translateY(2px);
    }

    .pill-cyan {
      background: linear-gradient(180deg, #22d3ee 0%, #0891b2 100%);
      color: white;
      box-shadow: 0 3px 0 #0e7490, 0 6px 15px rgba(6, 182, 212, 0.3);
    }
    .pill-purple {
      background: linear-gradient(180deg, #c084fc 0%, #7e22ce 100%);
      color: white;
      box-shadow: 0 3px 0 #6b21a8, 0 6px 15px rgba(168, 85, 247, 0.3);
    }
    .pill-pink {
      background: linear-gradient(180deg, #f472b6 0%, #db2777 100%);
      color: white;
      box-shadow: 0 3px 0 #be185d, 0 6px 15px rgba(236, 72, 153, 0.35);
    }
    .pill-blue {
      background: linear-gradient(180deg, #38bdf8 0%, #0284c7 100%);
      color: white;
      box-shadow: 0 3px 0 #0369a1, 0 6px 15px rgba(2, 132, 199, 0.3);
    }
    .pill-emerald {
      background: linear-gradient(180deg, #34d399 0%, #059669 100%);
      color: white;
      box-shadow: 0 3px 0 #047857, 0 6px 15px rgba(168, 85, 247, 0.25);
    }

    .btn-render-3d {
      background: linear-gradient(135deg, #0284c7 0%, #0ea5e9 50%, #38bdf8 100%);
      color: white;
      font-weight: 800;
      border-radius: 20px;
      box-shadow: 0 6px 0 #0369a1, 0 12px 28px rgba(2, 132, 199, 0.35);
      border-top: 1px solid rgba(255, 255, 255, 0.4);
      transition: all 0.16s cubic-bezier(0.16, 1, 0.3, 1);
    }
    .btn-render-3d:hover {
      filter: brightness(1.08);
      transform: translateY(-2px);
      box-shadow: 0 8px 0 #0369a1, 0 16px 32px rgba(2, 132, 199, 0.45);
    }
    .btn-render-3d:active {
      transform: translateY(4px);
      box-shadow: 0 2px 0 #0369a1, 0 4px 10px rgba(2, 132, 199, 0.3);
    }

    /* سوییچ سریع و روان خورشید و ماه */
    .theme-toggle-btn {
      position: relative;
      width: 64px;
      height: 34px;
      border-radius: 9999px;
      background: var(--card-white);
      border: 1px solid var(--glass-border);
      cursor: pointer;
      display: flex;
      align-items: center;
      padding: 3px;
    }
    .theme-toggle-knob {
      width: 28px;
      height: 28px;
      border-radius: 9999px;
      background: linear-gradient(135deg, #f59e0b, #fbbf24);
      display: flex;
      align-items: center;
      justify-content: center;
      color: #78350f;
      font-size: 13px;
      transform: translate3d(0, 0, 0);
      transition: transform 0.24s cubic-bezier(0.34, 1.56, 0.64, 1), background 0.2s ease;
      box-shadow: 0 2px 8px rgba(0,0,0,0.2);
    }
    html[data-theme="dark"] .theme-toggle-knob {
      transform: translate3d(30px, 0, 0);
      background: linear-gradient(135deg, #38bdf8, #818cf8);
      color: white;
    }
  </style>
</head>
<body class="p-3 md:p-8">

  <div class="max-w-7xl mx-auto space-y-6">

    <!-- Top Navigation Header -->
    <div class="frosted-panel p-4 md:px-7 flex items-center justify-between">
      <div class="flex items-center gap-3.5">
        <div class="w-11 h-11 rounded-2xl flex items-center justify-center text-white text-xl shadow-md" style="background: var(--accent-grad);">
          <i class="fa-solid fa-gem"></i>
        </div>
        <div class="flex items-center gap-3">
          <h1 class="text-xl md:text-2xl font-black tracking-wider uppercase bg-clip-text text-transparent" style="background-image: var(--accent-grad);">
            NEOTENET
          </h1>
          <span class="text-xs font-bold px-3 py-1 rounded-full border text-cyan-600 border-cyan-400/40 shadow-sm" style="background: var(--card-white);">
            QWEN STUDIO
          </span>
        </div>
      </div>

      <button class="theme-toggle-btn" onclick="toggleTheme()" title="Switch Day/Night Mode">
        <div class="theme-toggle-knob" id="theme_icon">
          <i class="fa-solid fa-sun"></i>
        </div>
      </button>
    </div>

    <!-- Main Workspace Grid -->
    <div class="grid grid-cols-1 lg:grid-cols-12 gap-6">

      <!-- LEFT COLUMN: CONTROLS (5 Cols) -->
      <div class="lg:col-span-5 space-y-5">

        <!-- CONDITIONING SLOTS -->
        <div class="frosted-panel p-5 space-y-4">
          <div class="flex justify-between items-center">
            <label class="text-xs font-bold uppercase tracking-wider flex items-center gap-2 text-sky-600">
              <i class="fa-solid fa-layer-group"></i>
              <span>Conditioning Slots (1 to 10)</span>
            </label>
            <button type="button" id="btn_add_slot" onclick="addDonorSlot()" class="btn-pill-3d pill-cyan text-xs">
              <i class="fa-solid fa-plus text-[10px]"></i>
              <span>Add Donor</span>
            </button>
          </div>
          <div class="space-y-3.5 max-h-[380px] overflow-y-auto pr-1" id="slots_grid"></div>
        </div>

        <!-- EDITING DIRECTIVE -->
        <div class="frosted-panel p-5 space-y-3.5">
          <div class="flex justify-between items-center">
            <label class="text-xs font-bold uppercase tracking-wider flex items-center gap-2 text-pink-500">
              <i class="fa-solid fa-pen-nib"></i>
              <span>Directive Instruction</span>
            </label>
            <div class="flex items-center gap-1.5 text-[11px]">
              <span class="px-2.5 py-0.5 rounded-lg font-bold bg-sky-500/15 text-sky-600 border border-sky-400/30">Image 1</span>
              <span style="color: var(--text-sub);">Canvas</span>
              <span style="color: var(--text-sub);">&bull;</span>
              <span class="px-2.5 py-0.5 rounded-lg font-bold bg-purple-500/15 text-purple-600 border border-purple-400/30">Image 2</span>
              <span style="color: var(--text-sub);">Donor</span>
            </div>
          </div>

          <textarea id="prompt_input" rows="4"
            placeholder="Type your instruction freely... e.g. Put the clothes from <image2> on <image1>, keep face and pose intact."
            class="w-full rounded-2xl p-3.5 text-xs focus:outline-none transition-all duration-200 resize-none border shadow-inner"
            style="background: var(--card-white); border-color: var(--card-border); color: var(--text-main);"></textarea>

          <!-- دکمه‌های کپسولی ۳ بعدی رنگی -->
          <div class="flex flex-wrap gap-2 pt-1">
            <button type="button" onclick="insertTag('<image1>')" class="btn-pill-3d pill-cyan">
              <i class="fa-solid fa-plus text-[10px]"></i> Image 1
            </button>
            <button type="button" onclick="insertTag('<image2>')" class="btn-pill-3d pill-purple">
              <i class="fa-solid fa-plus text-[10px]"></i> Image 2
            </button>
            <button type="button" onclick="insertPhrase('Keep the face, pose, and background intact.')" class="btn-pill-3d pill-blue">
              <i class="fa-solid fa-lock text-[10px]"></i> Lock Face/Pose
            </button>
            <button type="button" onclick="insertPhrase('fully naked, detailed bare chest, detailed nipples, natural skin texture, uncensored')" class="btn-pill-3d pill-pink">
              <i class="fa-solid fa-wand-magic-sparkles text-[10px]"></i> Detailed Anatomy
            </button>
            <button type="button" onclick="insertPhrase('Remove the background, and output a PNG image with transparent alpha background')" class="btn-pill-3d pill-emerald">
              <i class="fa-solid fa-shapes text-[10px]"></i> Transparent PNG
            </button>
          </div>
        </div>

        <!-- PARAMETER SLIDERS (مقادیر پیش‌فرض دقیقاً مطابق با کد شما) -->
        <div class="frosted-panel p-5 space-y-4 text-xs">
          <div>
            <div class="flex justify-between mb-1.5">
              <span class="font-bold flex items-center gap-1.5">Sampling Steps</span>
              <span id="steps_val" class="text-sky-600 font-mono font-bold">20</span>
            </div>
            <input type="range" id="steps_slider" min="4" max="40" value="20" oninput="document.getElementById('steps_val').innerText = this.value" class="w-full accent-sky-500">
          </div>
          <div>
            <div class="flex justify-between mb-1.5">
              <span class="font-bold flex items-center gap-1.5">Guidance CFG Scale</span>
              <span id="cfg_val" class="text-pink-500 font-mono font-bold">1.0</span>
            </div>
            <input type="range" id="cfg_slider" min="1.0" max="3.0" step="0.1" value="1.0" oninput="document.getElementById('cfg_val').innerText = parseFloat(this.value).toFixed(1)" class="w-full accent-pink-500">
          </div>
          <div class="grid grid-cols-2 gap-3.5 pt-1">
            <div>
              <label class="block mb-1.5 font-bold" style="color: var(--text-main);">Resolution Budget</label>
              <select id="res_budget" class="w-full rounded-xl p-2.5 text-xs border focus:outline-none shadow-sm font-medium" style="background: var(--card-white); border-color: var(--card-border); color: var(--text-main);">
                <option value="1024" selected>1 MP Budget (Fast)</option>
                <option value="0">Native Source Aspect</option>
                <option value="2048">Native 2K Output</option>
              </select>
            </div>
            <div>
              <label class="block mb-1.5 font-bold" style="color: var(--text-main);">Denoise Intensity</label>
              <input type="number" id="denoise_input" value="1.0" step="0.05" min="0.4" max="1.0" class="w-full rounded-xl p-2.5 text-xs border focus:outline-none shadow-sm font-medium" style="background: var(--card-white); border-color: var(--card-border); color: var(--text-main);">
            </div>
          </div>
        </div>

        <!-- DISPATCH BUTTON 3D -->
        <button id="btn_generate" onclick="runGeneration()" class="w-full py-4 btn-render-3d text-sm tracking-wider flex items-center justify-center gap-2 cursor-pointer">
          <i class="fa-solid fa-bolt"></i>
          <span>RENDER EDIT COMPOSITION</span>
        </button>
      </div>

      <!-- RIGHT COLUMN: VIEWPORT (7 Cols) -->
      <div class="lg:col-span-7 space-y-5">
        <!-- LIVE PROGRESS PANEL -->
        <div id="progress_card" class="hidden frosted-panel p-5">
          <div class="flex justify-between items-center text-xs mb-2.5">
            <span id="progress_title" class="font-bold flex items-center gap-2 text-sky-600">
              <i class="fa-solid fa-circle-notch fa-spin"></i>
              <span>Conditioning Diffusion Engine...</span>
            </span>
            <span id="progress_pct" class="font-mono font-bold text-pink-500">0%</span>
          </div>
          <div class="h-2.5 rounded-full overflow-hidden" style="background: rgba(0,0,0,0.06);">
            <div id="progress_bar" class="h-full rounded-full transition-all duration-200" style="background: var(--accent-grad); width: 0%;"></div>
          </div>
        </div>

        <!-- STAGE CONTAINER -->
        <div class="frosted-panel p-6 space-y-4">
          <div class="flex justify-between items-center">
            <h3 class="text-sm font-bold flex items-center gap-2">
              <i class="fa-solid fa-shapes text-sky-600"></i>
              <span>Composition Viewport</span>
            </h3>
            <div class="flex items-center space-x-2">
              <span id="alpha_badge" class="hidden px-3 py-1 rounded-full text-xs font-mono font-bold bg-purple-500/15 text-purple-600 border border-purple-400/30">RGBA Alpha</span>
              <span id="timer_badge" class="hidden px-3 py-1 rounded-full text-xs font-mono font-bold bg-emerald-500/15 text-emerald-600 border border-emerald-400/30">Finished</span>
            </div>
          </div>

          <!-- RESULT IMAGE VIEWPORT -->
          <div class="rounded-2xl p-4 text-center relative min-h-[460px] flex items-center justify-center overflow-hidden border shadow-inner" style="background: var(--card-white); border-color: var(--card-border);">
            <div id="result_placeholder" class="text-center space-y-2.5 p-8">
              <div class="w-16 h-16 mx-auto rounded-3xl flex items-center justify-center text-3xl shadow-sm border" style="background: var(--glass-panel); border-color: var(--glass-border); color: var(--text-sub);">
                <i class="fa-regular fa-image"></i>
              </div>
              <p class="text-xs font-medium" style="color: var(--text-sub);">Upload your images, describe your directive on the left, and click Render.</p>
            </div>
            <img id="stage_result" class="hidden max-h-[500px] w-auto mx-auto object-contain rounded-xl shadow-2xl transition-all duration-300">
          </div>

          <div id="download_bar" class="hidden flex justify-end pt-1">
            <a id="btn_download" download="neo_tenet_edit.png" class="px-5 py-2.5 btn-pill-3d pill-blue text-xs">
              <i class="fa-solid fa-download"></i>
              <span>Download High-Res PNG</span>
            </a>
          </div>
        </div>
      </div>

    </div>
  </div>

  <script>
    let donorSlots = ['portrait_model_denim.png'];

    function toggleTheme() {
      const html = document.documentElement;
      const isDark = html.getAttribute('data-theme') === 'dark';
      const newTheme = isDark ? 'light' : 'dark';
      html.setAttribute('data-theme', newTheme);

      const icon = document.getElementById('theme_icon');
      icon.style.transform = 'scale(0.7) rotate(90deg)';
      setTimeout(() => {
        icon.innerHTML = newTheme === 'light' ? '<i class="fa-solid fa-sun"></i>' : '<i class="fa-solid fa-moon"></i>';
        icon.style.transform = 'scale(1) rotate(0deg)';
      }, 100);
    }

    function renderSlotsUI() {
      const container = document.getElementById('slots_grid');
      container.innerHTML = '';
      donorSlots.forEach((filename, idx) => {
        const slotNum = idx + 1;
        const isCanvas = slotNum === 1;
        const card = document.createElement('div');
        card.className = "frosted-card p-4 border transition-all duration-200 flex items-center gap-4 relative shadow-sm";

        card.innerHTML = `
          <div class="relative w-20 h-20 md:w-24 md:h-24 rounded-2xl overflow-hidden border flex items-center justify-center flex-shrink-0 shadow-sm" style="background: rgba(0,0,0,0.04); border-color: var(--card-border);">
            ${filename ? `<img src="/input/${filename}" class="w-full h-full object-cover">` : `<i class="fa-solid fa-cloud-arrow-up text-2xl" style="color: var(--text-sub);"></i>`}
          </div>
          <div class="flex-1 space-y-2">
            <div class="flex justify-between items-center">
              <div class="flex items-center gap-2">
                <span class="px-2.5 py-0.5 rounded-lg text-xs font-black font-mono shadow-sm ${isCanvas ? 'bg-sky-500/20 text-sky-600 border border-sky-400/40' : 'bg-purple-500/20 text-purple-600 border border-purple-400/40'}">
                  Image ${slotNum}
                </span>
                <span class="text-xs font-semibold" style="color: var(--text-sub);">${isCanvas ? '(Primary Canvas)' : '(Donor Ref)'}</span>
              </div>
              ${!isCanvas ? `<button onclick="removeDonorSlot(${idx})" class="text-red-400 hover:text-red-500 text-xs p-1"><i class="fa-solid fa-trash"></i></button>` : ''}
            </div>

            <p class="text-[11px]" style="color: var(--text-sub);">${isCanvas ? 'Anchors pose, perspective & face.' : 'Attributes donor image.'}</p>

            <div class="pt-0.5 flex items-center gap-2.5">
              <label class="btn-pill-3d pill-blue">
                <i class="fa-solid fa-arrow-up-from-bracket text-[11px]"></i>
                <span>Choose Image</span>
                <input type="file" accept="image/*" class="hidden" onchange="uploadFile(${idx}, this.files[0])">
              </label>
              <span class="text-[10px] font-medium truncate max-w-[130px]" style="color: var(--text-sub);">${filename ? filename.split('_').slice(-1)[0] : 'No file chosen'}</span>
            </div>
          </div>
        `;
        container.appendChild(card);
      });
    }

    function addDonorSlot() {
      if (donorSlots.length >= 10) return alert("Maximum 10 reference slots reached.");
      donorSlots.push(null);
      renderSlotsUI();
    }

    function removeDonorSlot(i) {
      donorSlots.splice(i, 1);
      renderSlotsUI();
    }

    async function uploadFile(index, file) {
      if (!file) return;
      const fd = new FormData();
      fd.append('file', file);
      const res = await fetch('/api/upload', { method: 'POST', body: fd });
      const data = await res.json();
      donorSlots[index] = data.filename;
      renderSlotsUI();
    }

    function insertTag(tag) {
      const el = document.getElementById('prompt_input');
      el.value += (el.value ? ' ' : '') + tag;
      el.focus();
    }

    function insertPhrase(phrase) {
      const el = document.getElementById('prompt_input');
      el.value += (el.value ? ', ' : '') + phrase;
      el.focus();
    }

    function runGeneration() {
      const prompt = document.getElementById('prompt_input').value.trim();
      if (!prompt) return alert("Please enter directive instruction!");
      const validSlots = donorSlots.filter(s => s !== null);
      if (validSlots.length === 0) return alert("Please upload at least <image1>!");

      // داده‌ها دقیقاً با همان ساختار و مقادیر اصلی کد شما ارسال می‌شوند
      const payload = {
        instruction: prompt,
        slots: validSlots,
        steps: parseInt(document.getElementById('steps_slider').value),
        cfg: parseFloat(document.getElementById('cfg_slider').value),
        resolution: parseInt(document.getElementById('res_budget').value),
        denoise: parseFloat(document.getElementById('denoise_input').value)
      };

      document.getElementById('progress_card').classList.remove('hidden');
      document.getElementById('progress_bar').style.width = '0%';
      document.getElementById('progress_pct').innerText = '0%';
      document.getElementById('progress_title').innerText = '🧠 Reading & Analyzing Vision-Language (Qwen3-VL)...';
      document.getElementById('btn_generate').disabled = true;

      const evtSource = new EventSource('/api/generate_stream?payload=' + encodeURIComponent(JSON.stringify(payload)));

      const nodeLabels = {
        '459:474': '🧠 Reading Images & Prompt (Qwen3-VL)...',
        '459:458': '⚡ Synthesizing & Sampling Composition...',
        '459:457': '🎨 Decoding Final Image (4-Channel VAE)...',
        '459:451': '📦 Loading AI Diffusion Weights...',
        '459:453': '📦 Loading Vision Encoder...',
        '459:454': '📦 Loading 4-Channel VAE...',
        '461': '💾 Saving Final High-Res PNG...'
      };

      evtSource.onmessage = function(e) {
        const msg = JSON.parse(e.data);
        if (msg.type === 'progress') {
          const pct = Math.round((msg.value / msg.max) * 100);
          document.getElementById('progress_bar').style.width = pct + '%';
          document.getElementById('progress_pct').innerText = pct + '%';
          document.getElementById('progress_title').innerText = `⚡ Denoising Step ${msg.value} / ${msg.max} (${pct}%)`;
        } else if (msg.type === 'executing') {
          if (msg.node) {
            const label = nodeLabels[msg.node] || ('⚙️ Processing AI Pipeline...');
            document.getElementById('progress_title').innerText = label;
          }
        } else if (msg.type === 'done') {
          evtSource.close();
          document.getElementById('progress_card').classList.add('hidden');
          document.getElementById('result_placeholder').classList.add('hidden');
          const img = document.getElementById('stage_result');
          img.src = 'data:image/png;base64,' + msg.image_b64;
          img.classList.remove('hidden');
          if (msg.has_alpha) document.getElementById('alpha_badge').classList.remove('hidden');
          document.getElementById('download_bar').classList.remove('hidden');
          document.getElementById('btn_download').href = img.src;
          document.getElementById('btn_generate').disabled = false;
        }
      };

      evtSource.onerror = function() {
        evtSource.close();
        document.getElementById('btn_generate').disabled = false;
        alert("Generation stopped or server connection closed.");
      };
    }

    renderSlotsUI();
  </script>
</body>
</html>'''

@app.route('/')
def index():
    return HTML_TEMPLATE

@app.route('/input/<path:filename>')
def serve_input(filename):
    return send_from_directory('/content/ComfyUI/input', filename)

@app.route('/api/upload', methods=['POST'])
def upload_endpoint():
    if 'file' not in request.files:
        return jsonify({'error': 'No file part'}), 400
    f = request.files['file']
    fname = f"{uuid.uuid4().hex[:8]}_{f.filename}"
    f.save(os.path.join('/content/ComfyUI/input', fname))
    print(f"\n[Storage] 📤 Asset Saved: {fname}", flush=True)
    return jsonify({'filename': fname})

@app.route('/api/generate_stream')
def generate_stream():
    params = json.loads(request.args.get('payload'))
    client_id = str(uuid.uuid4())
    raw_slots = params.get('slots', ['portrait_model_denim.png'])[:10]
    num_slots = len(raw_slots)

    optimized_slots = []
    for idx, slot_file in enumerate(raw_slots, 1):
        if idx == 1:
            optimized_slots.append(slot_file)
        else:
            optimized_slots.append(optimize_donor_image(slot_file, max_dim=768))

    print("\n" + "═" * 65, flush=True)
    print(f"[NeoTenet] 📥 Task Received ({num_slots} Active Slots)", flush=True)
    print(f"• Directive : {params['instruction'][:85]}...", flush=True)
    print(f"• Config    : Steps={params['steps']} | CFG={params['cfg']} | Denoise={params['denoise']} | Budget={params['resolution']}", flush=True)
    print("═" * 65, flush=True)

    # ساختار دست‌نخورده و ۱۰۰٪ پایدار ورک‌فلو اورجینال شما
    workflow = {
      "461": {"inputs": {"filename_prefix": "NeoTenet_Edit", "format": "png", "format.bit_depth": "8-bit", "format.input_color_space": "sRGB", "images": ["459:457", 0]}, "class_type": "SaveImageAdvanced"},
      "459:451": {"inputs": {"unet_name": "qwen_image_2.1_int8_convrot.safetensors", "weight_dtype": "default"}, "class_type": "UNETLoader"},
      "459:453": {"inputs": {"clip_name": "qwen3vl_8b_int8_convrot.safetensors", "type": "qwen_image", "device": "default"}, "class_type": "CLIPLoader"},
      "459:454": {"inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}, "class_type": "VAELoader"},
      "459:456": {"inputs": {"width": 1024, "height": 1024, "batch_size": 1}, "class_type": "EmptyLatentImage"},
      "459:457": {"inputs": {"samples": ["459:458", 0], "vae": ["459:454", 0]}, "class_type": "VAEDecode"},
      "459:458": {
        "inputs": {
          "seed": int(time.time() * 1000) % 1000000000,
          "steps": params['steps'],
          "cfg": params['cfg'],
          "sampler_name": "euler",
          "scheduler": "simple",
          "denoise": params['denoise'],
          "model": ["459:469", 0],
          "positive": ["459:474", 0],
          "negative": ["459:474", 1],
          "latent_image": ["459:468", 0]
        },
        "class_type": "KSampler"
      },
      "459:468": {"inputs": {"switch": False, "on_false": ["459:474", 2], "on_true": ["459:456", 0]}, "class_type": "ComfySwitchNode"},
      "459:469": {"inputs": {"device": "auto", "dtype": "int8", "model": ["459:451", 0]}, "class_type": "QwenImage21Cache"},
      "459:474": {
        "inputs": {
          "prompt": ["459:484", 0],
          "negative_prompt": "" if params['cfg'] <= 1.0 else "blurry, low quality, artifacts, distorted features",
          "resolution": params['resolution'],
          "clip": ["459:453", 0],
          "vae": ["459:454", 0]
        },
        "class_type": "TextEncodeQwenImage21"
      },
      "459:484": {"inputs": {"switch": False, "on_false": params['instruction'], "on_true": params['instruction']}, "class_type": "ComfySwitchNode"}
    }

    for idx, slot_file in enumerate(optimized_slots, 1):
        node_id = f"slot_node_{idx}"
        workflow[node_id] = {
            "inputs": {"image": slot_file},
            "class_type": "LoadImage"
        }
        workflow["459:474"]["inputs"][f"images.image_{idx}"] = [node_id, 0]

    def event_stream():
        ws = websocket.WebSocket()
        ws.connect(f"ws://127.0.0.1:8188/ws?clientId={client_id}")

        req_data = json.dumps({"prompt": workflow, "client_id": client_id}).encode('utf-8')
        req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=req_data)
        with urllib.request.urlopen(req) as resp:
            job_meta = json.loads(resp.read())
        job_id = job_meta['prompt_id']
        start_t = time.time()
        print(f"[NeoTenet] 🚀 Prompt Dispatched (Job ID: {job_id})", flush=True)

        while True:
            msg = ws.recv()
            if not isinstance(msg, str):
                continue
            ev = json.loads(msg)
            ev_type = ev.get('type')
            data = ev.get('data', {})

            if ev_type == 'progress':
                val = data.get('value', 0)
                total = data.get('max', params['steps'])
                pct = int((val / total) * 100) if total > 0 else 0
                print(f"\r[NeoTenet] ⚡ Sampling Step {val}/{total} ({pct}%)", end="", flush=True)
                yield f"data: {json.dumps({'type': 'progress', 'value': val, 'max': total})}\n\n"
            elif ev_type == 'executing':
                node = data.get('node')
                if node is None and data.get('prompt_id') == job_id:
                    break
                if node:
                    yield f"data: {json.dumps({'type': 'executing', 'node': node})}\n\n"

        ws.close()
        dur = time.time() - start_t

        outs = glob.glob('/content/ComfyUI/output/NeoTenet_Edit*.png')
        latest = max(outs, key=os.path.getctime)
        with open(latest, 'rb') as f:
            b64_str = base64.b64encode(f.read()).decode('utf-8')
        with Image.open(latest) as img:
            is_alpha = img.mode == 'RGBA'

        print(f"\n[NeoTenet] ✓ Completed in {dur:.1f}s | Output: {os.path.basename(latest)}\n" + "═" * 65, flush=True)
        yield f"data: {json.dumps({'type': 'done', 'image_b64': b64_str, 'duration': dur, 'has_alpha': is_alpha})}\n\n"

    return Response(event_stream(), mimetype="text/event-stream")

threading.Thread(target=lambda: app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False), daemon=True).start()
time.sleep(2)

print("═" * 65)
print("💎 STARTING NEOTENET HTTPS PUBLIC TUNNEL")
print("═" * 65)

tunnel = pinggy.start_tunnel(forwardto="localhost:5000")

public_url = None
for _ in range(20):
    if hasattr(tunnel, "urls") and tunnel.urls:
        for u in tunnel.urls:
            if u.startswith("https://") and "dashboard" not in u:
                public_url = u
                break
    if public_url:
        break
    time.sleep(1)

# کادر شیک و مرتب بدون روی هم افتادن متن در موبایل
if public_url:
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, rgba(8, 20, 38, 0.95), rgba(16, 36, 68, 0.95)); backdrop-filter: blur(20px); border: 1.5px solid rgba(56, 189, 248, 0.4); border-radius: 24px; padding: 24px; text-align: center; margin: 20px auto; max-width: 500px; box-shadow: 0 15px 35px rgba(0,0,0,0.4);">
      <div style="display: inline-flex; align-items: center; justify-content: center; width: 52px; height: 52px; border-radius: 16px; background: linear-gradient(135deg, #0284c7, #38bdf8); color: white; font-size: 24px; margin-bottom: 12px; box-shadow: 0 4px 15px rgba(56,189,248,0.4);">
        💎
      </div>
      <h2 style="color: #38bdf8; font-size: 22px; font-weight: 900; letter-spacing: 1px; margin: 0 0 6px 0; font-family: ui-sans-serif, system-ui;">NEOTENET STUDIO IS READY</h2>
      <p style="color: #94a3b8; font-size: 13px; margin: 0 0 20px 0;">Frosted Glass Dashboard Edition &bull; Dual-RAM Stable Engine</p>

      <div style="margin: 0 auto; display: flex; justify-content: center;">
        <a href="{public_url}" target="_blank" style="display: inline-block; background: linear-gradient(135deg, #0284c7 0%, #06b6d4 100%); color: #ffffff; padding: 14px 32px; border-radius: 16px; text-decoration: none; font-weight: 800; font-size: 15px; box-shadow: 0 8px 24px rgba(6, 182, 212, 0.4); border: 1px solid rgba(255,255,255,0.2);">
          🚀 Open NeoTenet Studio
        </a>
      </div>
      <div style="margin-top: 14px; font-size: 11px; font-family: monospace; color: #64748b; word-break: break-all;">
        {public_url}
      </div>
    </div>
    """))

print("\n" + "═" * 65)
print("📡 LIVE TASK & SAMPLING LOGS (Waiting for task from web interface...)")
print("═" * 65 + "\n", flush=True)

 * Serving Flask app '__main__'
 * Debug mode: off
═════════════════════════════════════════════════════════════════
💎 STARTING NEOTENET HTTPS PUBLIC TUNNEL
═════════════════════════════════════════════════════════════════



═════════════════════════════════════════════════════════════════
📡 LIVE TASK & SAMPLING LOGS (Waiting for task from web interface...)
═════════════════════════════════════════════════════════════════

